In [1]:
%matplotlib qt

In [2]:
from turtle import pos

import matplotlib

# matplotlib.use("MacOSX")  # for mac users

# print(matplotlib.get_backend())

import sys

import matplotlib as mpl

sys.path.append("/Users/yao/Smilei")
import happi
import matplotlib.pyplot as plt
import numpy as np

jetcmap = plt.cm.get_cmap(
    "jet", 9
)  # generate a jet map with 10 values "rainbow", "jet", YlOrRd
jet_vals = jetcmap(np.arange(9))  # extract those values as an array
jet_vals[0] = [1.0, 1, 1.0, 1]  # change the first value
jet_vals[8] = [0.0, 0, 0.0, 1]  # change the first value
newcmap = mpl.colors.LinearSegmentedColormap.from_list("mine", jet_vals)

from matplotlib import font_manager

font_dirs = ["/Users/yao/Documents/Calibri and Cambria Fonts/"]
font_files = font_manager.findSystemFonts(fontpaths=font_dirs)

for font_file in font_files:
    font_manager.fontManager.addfont(font_file)

# set font
plt.rcParams["font.family"] = "Calibri"

plt.rc("text", usetex=False)
plt.rc("xtick", labelsize=14)
plt.rc("ytick", labelsize=14)
plt.rc("axes", labelsize=14)
plt.rc("legend", fontsize=12)

import scienceplots as splt

plt.style.use(["nature", "notebook", "grid", "high-vis"])

/var/folders/2t/97rc3fl92tg15k2l_4sk5hsh0000gn/T/ipykernel_77629/2227518788.py:18: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  jetcmap = plt.cm.get_cmap(


In [33]:
wkdir = [
    # "/Users/yao/Documents/Data/IFE/LG_data1/LG_l1_Ltot2_CP_a600_avB/",
    "/Volumes/Irene/Sim_IFE_Rome/SG_CP_l1_Ltot0_longVacuum_largeLy/"
]

S0 = happi.Open(
    wkdir[0], reference_angular_frequency_SI=2.0 * np.pi * 3e8 / (1.0 * 1e-6)
)

# Get simulation box size
Lx = S0.namelist.Lx / 2 / np.pi  # in um
print("Lx = ", Lx)
Ly = S0.namelist.Ly / 2 / np.pi  # in um
print("Ly = ", Ly)
Lz = S0.namelist.Lz / 2 / np.pi  # in um
print("Lz = ", Lz)

Loaded simulation '/Volumes/Irene/Sim_IFE_Rome/SG_CP_l1_Ltot0_longVacuum_largeLy/'
Scanning for Scalar diagnostics
Scanning for Field diagnostics
Scanning for Probe diagnostics
Scanning for ParticleBinning diagnostics
Scanning for RadiationSpectrum diagnostics
Scanning for Performance diagnostics
Scanning for Screen diagnostics
Scanning for Tracked particle diagnostics
Scanning for new particle diagnostics
Lx =  80.0
Ly =  44.8
Lz =  44.8


In [43]:
S0.Probe(0, 'Bx',units=["um","fs","1e5 T * 100 fs"],
        #  vsym=True,
         vsym=2,
         cmap='bwr').slide()

Probe #0: 2-dimensional, with fields Ex,Ey,Ez,Bx,By,Bz,Rho_eon,Rho_ion,Jx_eon,Jx_ion,Jy_eon,Jy_ion,Jz_eon,Jz_ion
	p0 = 0.0 0.0 140.74335088082273
	p1 = 502.6548245743669 0.0 140.74335088082273
	p2 = 0.0 281.48670176164546 140.74335088082273
	number = 800 448


In [40]:
S0.Field(0, 'Bx',units=["um","fs","1e5 T * 100 fs", "1e5 T"],
         vsym=True,
         subset={'z':Ly/2.},
         cmap='bwr').slide()

Field diagnostic #0: Bx
	Grid spacing: 2.5132741228718345, 2.5132741228718345, 2.5132741228718345
	subset at z = [22.61946711] L_r


In [5]:
S0.Probe(1, 'Bx',units=["um","fs","1e5 T * 100 fs"],
         vsym=True,
         cmap='bwr').slide()

Probe #1: 2-dimensional, with fields Ex,Ey,Ez,Bx,By,Bz,Rho_ion,Rho_eon,Jx_ion,Jy_ion,Jz_ion,Jx_eon,Jy_eon,Jz_eon
	p0 = 62.83185307179586 0.0 0.0
	p1 = 62.83185307179586 150.79644737231007 0.0
	p2 = 62.83185307179586 0.0 150.79644737231007
	number = 299 240


In [6]:
## now, instead of the FFT noise cancelation, we will focus more on the interactive average of the Bx field map;
## in a case-by-case manner. 

def get_Bx_data(S, time):
    Bx = S.Probe(0, "Bx", units=["um", "fs", "1e5 T * 100 fs"])
    data = np.array(Bx.getData()[time])
    field_x = np.array(Bx.getAxis("axis1")[:,0])
    field_y = np.array(Bx.getAxis("axis2")[:,1])
    print("time = ", Bx.getTimes()[time], " fs")
    return data, field_x, field_y

In [7]:
data, field_x, field_y = get_Bx_data(S0, 50)

time =  164.5448267190433  fs


In [8]:
## functions to do the spatial average of Bx over a certain region at a given timestep

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Button
from matplotlib.patches import Rectangle, Circle

import happi


def get_Bx_data(wkdir, time_index, lambda0_um=1.0):
    S = happi.Open(
        wkdir,
        reference_angular_frequency_SI=2.0 * np.pi * 3e8 / (lambda0_um * 1e-6)
    )
    Bx = S.Probe(0, "Bx", units=["um", "fs", "1e5 T * 100 fs"])
    data = np.array(Bx.getData()[time_index])
    field_x = np.array(Bx.getAxis("axis1")[:, 0])
    field_y = np.array(Bx.getAxis("axis2")[:, 1])
    t_fs_int = int(np.rint(float(Bx.getTimes()[time_index])))
    return data, field_x, field_y, t_fs_int


def ensure_increasing_axes(B, x, y):
    B2, x2, y2 = B, x, y
    if x2[0] > x2[-1]:
        x2 = x2[::-1]
        B2 = B2[:, ::-1]
    if y2[0] > y2[-1]:
        y2 = y2[::-1]
        B2 = B2[::-1, :]
    return B2, x2, y2


def stats_from_indices(B, iy0, iy1, ix0, ix1):
    sub = B[iy0:iy1, ix0:ix1]
    mean = np.nanmean(sub)
    std  = np.nanstd(sub)
    vmin = np.nanmin(sub)
    vmax = np.nanmax(sub)
    n    = int(np.sum(np.isfinite(sub)))
    return mean, std, vmin, vmax, n


def stats_circle(B, icx, icy, r_cells):
    ny, nx = B.shape
    yy, xx = np.ogrid[:ny, :nx]
    mask = (xx - icx)**2 + (yy - icy)**2 <= r_cells**2
    vals = B[mask]
    mean = np.nanmean(vals)
    std  = np.nanstd(vals)
    vmin = np.nanmin(vals)
    vmax = np.nanmax(vals)
    n    = int(np.sum(np.isfinite(vals)))
    return mean, std, vmin, vmax, n


class ROIAverager:
    """
    fixed_mode: "circle" / "rect" / "square" (click-to-place + arrow-key nudging)
    """
    def __init__(
        self, B, x_um, y_um,
        wkdir,
        t_fs_int,
        cmap="seismic",
        vmin=None, vmax=None,
        fixed_mode="circle",
        nx_cells=60, ny_cells=60,
        r_cells=30,
        title_prefix="Time-integrated Bx (1e5 T * 100 fs)",
    ):
        self.B = B
        self.x = x_um
        self.y = y_um
        self.wkdir = wkdir
        self.case_name = os.path.basename(os.path.normpath(wkdir))
        self.t_fs_int = int(t_fs_int)

        self.fixed_mode = fixed_mode
        self.nx_cells = int(nx_cells)
        self.ny_cells = int(ny_cells)
        self.r_cells = int(r_cells)

        # ROI center (indices). None until first click
        self.icx = None
        self.icy = None

        # --- figure ---
        self.fig, self.ax = plt.subplots(figsize=(8, 6))
        extent = [self.x.min(), self.x.max(), self.y.min(), self.y.max()]
        self.im = self.ax.imshow(
            self.B, origin="lower", extent=extent, aspect="auto",
            cmap=cmap, vmin=vmin, vmax=vmax
        )
        self.cb = self.fig.colorbar(self.im, ax=self.ax, pad=0.03)
        self.ax.set_xlabel("x (µm)")
        self.ax.set_ylabel("y (µm)")
        self.ax.set_title(f"{title_prefix} at t = {self.t_fs_int:d} fs")

        # Stats text on right, near colorbar
        self.stats_text = self.cb.ax.text(
            0.0, -0.08, "",
            transform=self.cb.ax.transAxes,
            ha="left", va="top", fontsize=11
        )

        # ROI patches
        self.roi_rect = Rectangle((0, 0), 0, 0, fill=False, lw=2)
        self.roi_circ = Circle((0, 0), radius=0, fill=False, lw=2)
        # requested: dashed white circle
        self.roi_circ.set_edgecolor("white")
        self.roi_circ.set_linestyle("--")

        self.ax.add_patch(self.roi_rect)
        self.ax.add_patch(self.roi_circ)
        self.roi_rect.set_visible(False)
        self.roi_circ.set_visible(False)

        # Buttons
        plt.subplots_adjust(bottom=0.16)
        self._btn_axes = []
        ax_btn_clear = self.fig.add_axes([0.08, 0.05, 0.18, 0.07])
        ax_btn_stats = self.fig.add_axes([0.30, 0.05, 0.18, 0.07])
        ax_btn_savef = self.fig.add_axes([0.52, 0.05, 0.18, 0.07])
        self._btn_axes.extend([ax_btn_clear, ax_btn_stats, ax_btn_savef])

        self.btn_clear = Button(ax_btn_clear, "Clear ROI")
        self.btn_save_stats = Button(ax_btn_stats, "Save stats")
        self.btn_save_fig = Button(ax_btn_savef, "Save figure")

        self.btn_clear.on_clicked(self.clear_roi)
        self.btn_save_stats.on_clicked(self.save_stats)
        self.btn_save_fig.on_clicked(self.save_figure)

        self.last_line = None

        # Events
        self.fig.canvas.mpl_connect("button_press_event", self.on_click_place)
        self.fig.canvas.mpl_connect("key_press_event", self.on_key)

    # ---------- helpers ----------
    def nearest_index(self, arr, val):
        return int(np.argmin(np.abs(arr - val)))

    def clamp_center(self):
        ny, nx = self.B.shape
        self.icx = int(np.clip(self.icx, 0, nx - 1))
        self.icy = int(np.clip(self.icy, 0, ny - 1))

    def update_stats_text(self, header, mean, std, vmin, vmax, n):
        msg = (
            f"{header}\n"
            f"mean = {mean:.6g}\n"
            f"std  = {std:.6g}\n"
            f"min  = {vmin:.6g}\n"
            f"max  = {vmax:.6g}\n"
            f"Npix = {n}"
        )
        self.stats_text.set_text(msg)
        self.fig.canvas.draw_idle()
        print(msg.replace("\n", " | "))

    def roi_tag(self):
        """ROI descriptor for filenames."""
        if self.fixed_mode == "circle":
            return f"circle_r{self.r_cells:03d}"
        if self.fixed_mode == "square":
            return f"square_n{self.nx_cells:03d}"
        if self.fixed_mode == "rect":
            return f"rect_nx{self.nx_cells:03d}_ny{self.ny_cells:03d}"
        return "roi"

    def center_tag(self):
        """Center indices tag for filenames."""
        if self.icx is None or self.icy is None:
            return "icxNA_icyNA"
        return f"icx{self.icx:04d}_icy{self.icy:04d}"

    def build_pdf_path(self):
        # Example:
        # case__Bx__t001234fs__circle_r030__icx0456_icy0321.pdf
        fname = f"{self.case_name}__Bx__t{self.t_fs_int:06d}fs__{self.roi_tag()}__{self.center_tag()}.pdf"
        return os.path.join(self.wkdir, fname)

    def build_stats_path(self):
        # keep one stats file per case
        return os.path.join(self.wkdir, f"{self.case_name}__roi_stats.txt")

    # ---------- ROI update ----------
    def apply_roi_at_current_center(self):
        """Update patch + stats based on (icx, icy)."""
        if self.icx is None or self.icy is None:
            return

        self.clamp_center()
        icx, icy = self.icx, self.icy
        ny, nx = self.B.shape

        if self.fixed_mode in ("square", "rect"):
            nx_c = self.nx_cells
            ny_c = self.ny_cells if self.fixed_mode == "rect" else self.nx_cells

            hx = nx_c // 2
            hy = ny_c // 2

            ix0 = max(icx - hx, 0)
            ix1 = min(ix0 + nx_c, nx)
            ix0 = max(ix1 - nx_c, 0)

            iy0 = max(icy - hy, 0)
            iy1 = min(iy0 + ny_c, ny)
            iy0 = max(iy1 - ny_c, 0)

            self.roi_circ.set_visible(False)
            self.roi_rect.set_visible(True)

            self.roi_rect.set_xy((self.x[ix0], self.y[iy0]))
            self.roi_rect.set_width(self.x[ix1 - 1] - self.x[ix0])
            self.roi_rect.set_height(self.y[iy1 - 1] - self.y[iy0])

            mean, std, vmin, vmax, n = stats_from_indices(self.B, iy0, iy1, ix0, ix1)
            header = f"ROI {self.fixed_mode} ({nx_c}×{ny_c} cells)\ncenter≈({self.x[icx]:.3f},{self.y[icy]:.3f}) µm"
            self.update_stats_text(header, mean, std, vmin, vmax, n)

            self.last_line = (
                f"{self.fixed_mode}\t"
                f"t_fs={self.t_fs_int}\t"
                f"icx={icx}\ticy={icy}\t"
                f"nx={nx_c}\tny={ny_c}\t"
                f"mean={mean}\tstd={std}\tmin={vmin}\tmax={vmax}\tNpix={n}\n"
            )

        elif self.fixed_mode == "circle":
            r = self.r_cells

            self.roi_rect.set_visible(False)
            self.roi_circ.set_visible(True)
            self.roi_circ.center = (self.x[icx], self.y[icy])

            dx = float(np.median(np.diff(self.x))) if len(self.x) > 1 else 1.0
            self.roi_circ.radius = r * dx

            mean, std, vmin, vmax, n = stats_circle(self.B, icx, icy, r)
            header = f"ROI circle (r={r} cells)\ncenter≈({self.x[icx]:.3f},{self.y[icy]:.3f}) µm"
            self.update_stats_text(header, mean, std, vmin, vmax, n)

            self.last_line = (
                f"circle\t"
                f"t_fs={self.t_fs_int}\t"
                f"icx={icx}\ticy={icy}\t"
                f"r={r}\t"
                f"mean={mean}\tstd={std}\tmin={vmin}\tmax={vmax}\tNpix={n}\n"
            )

    # ---------- events ----------
    def on_click_place(self, event):
        if event.inaxes != self.ax:
            return
        if event.xdata is None or event.ydata is None:
            return
        self.icx = self.nearest_index(self.x, event.xdata)
        self.icy = self.nearest_index(self.y, event.ydata)
        self.apply_roi_at_current_center()

    def on_key(self, event):
        # Move ROI by exactly 1 cell per keypress
        if self.icx is None or self.icy is None:
            return

        key = event.key
        if key in ("left", "right", "up", "down"):
            if key == "left":
                self.icx -= 1
            elif key == "right":
                self.icx += 1
            elif key == "down":
                self.icy -= 1
            elif key == "up":
                self.icy += 1

            self.apply_roi_at_current_center()

    # ---------- buttons ----------
    def clear_roi(self, event=None):
        self.roi_rect.set_visible(False)
        self.roi_circ.set_visible(False)
        self.stats_text.set_text("")
        self.last_line = None
        self.icx = None
        self.icy = None
        self.fig.canvas.draw_idle()

    def save_stats(self, event=None):
        if self.last_line is None:
            print("No ROI selected; nothing to save.")
            return
        os.makedirs(self.wkdir, exist_ok=True)
        out = self.build_stats_path()
        with open(out, "a") as f:
            f.write(self.last_line)
        print(f"Saved stats -> {out}: {self.last_line.strip()}")

    def save_figure(self, event=None):
        """
        Save a PDF (dpi=600) without button boxes.
        Keeps: plot, labels, title, colorbar, ROI overlay, and stats text.
        """
        os.makedirs(self.wkdir, exist_ok=True)
        out_pdf = self.build_pdf_path()

        # Hide buttons temporarily
        prev_vis = [ax.get_visible() for ax in self._btn_axes]
        for ax in self._btn_axes:
            ax.set_visible(False)

        # Render and save
        self.fig.canvas.draw()
        self.fig.savefig(out_pdf, dpi=600, bbox_inches="tight")
        print(f"Saved figure -> {out_pdf}")

        # Restore
        for ax, v in zip(self._btn_axes, prev_vis):
            ax.set_visible(v)
        self.fig.canvas.draw_idle()

    def show(self):
        print(f"Fixed ROI mode: {self.fixed_mode}")
        print("  - left-click to place ROI center")
        print("  - arrow keys move ROI by 1 cell")
        print("  - Save stats -> case__roi_stats.txt in wkdir")
        print("  - Save figure -> case__Bx__t******fs__ROI__icx_icy.pdf in wkdir")
        plt.show()


# -----------------------------
# Example usage
# -----------------------------
# wkdir = "/Users/yao/Documents/Data/IFE/ife_007_a0900_ne120_res80_ppc4"
# it = 8
# data, x_um, y_um, t_fs_int = get_Bx_data(wkdir, it, lambda0_um=1.0)
# data, x_um, y_um = ensure_increasing_axes(data, x_um, y_um)
#
# tool = ROIAverager(
#     data, x_um, y_um,
#     wkdir=wkdir,
#     t_fs_int=t_fs_int,
#     vmin=-2.0, vmax=2.0,
#     fixed_mode="circle",
#     r_cells=30
# )
# tool.show()

In [ ]:
## get the spatial region manually for each case and save the stats in a txt file for later summary and plotting.

wkdir = [
        #    "/Users/yao/Documents/Data/IFE/LG_data1/LG_l1_Ltot2_CP_a600_avB/",
        #    "/Users/yao/Documents/Data/IFE/LG_data1/LG_l1_Ltot2_CP_a600_avB_NoRR/",
        #    "/Users/yao/Documents/Data/IFE/LG_data1/LG_l1_Ltot2_CP_ne50_a600_avB/",
        #    "/Users/yao/Documents/Data/IFE/LG_data1/LG_l1_Ltot2_CP_ne50_a600_avB_NoRR/",
        #    "/Volumes/Irene/Sim_IFE_Rome/SG_CP_l1_Ltot0_longVacuum_largeLy/",
        #    "/Volumes/Irene/Sim_IFE_Rome/SG_CP_l1_Ltot0_longVacuum_largeLy_NoRR/",
        #    "/Volumes/Irene/Sim_IFE_Rome/SG_CP_l1_Ltot2_longVacuum_largeLy/",
        #    "/Volumes/Irene/Sim_IFE_Rome/SG_CP_l1_Ltot2_longVacuum_largeLy_NoRR/",
           "/Volumes/Irene/Sim_IFE_Rome/SG_l1LP_a0_500_ne_60_longVacuum_largeLy/"
         ]
# For LG_l1_Ltot2_CP_xxxxx
# it = 30 -> time =  99 fs
# it = 35 -> time = 115 fs
# it = 40 -> time = 132 fs
# it = 45 -> time = 148 fs
# it = 50 -> time = 165 fs
# it = 50

# For SG_CP_l1_Ltot0_longVacuum_largeLy_xxxx
# it =  90 -> time = 296 fs
# it = 100 -> time = 329 fs
# it = 110 -> time = 362 fs
it = 90
data, x_um, y_um, t_fs_int = get_Bx_data(wkdir[0], it, lambda0_um=1.0)
data, x_um, y_um = ensure_increasing_axes(data, x_um, y_um)

tool = ROIAverager(
    data.T, x_um, y_um,
    wkdir=wkdir[0],
    t_fs_int=t_fs_int,
    # vmin=-2.0, vmax=2.0, # For LG_l1_Ltot2_CP_xxxxx
    vmin=-5.0, vmax=5.0, # For SG_CP_l1_Ltot0_longVacuum_largeLy_xxxx
    fixed_mode="circle",
    r_cells=12
)
tool.show()

Loaded simulation '/Volumes/Irene/Sim_IFE_Rome/SG_l1LP_a0_500_ne_60_longVacuum_largeLy/'
Scanning for Scalar diagnostics
Scanning for Field diagnostics
Scanning for Probe diagnostics
Scanning for ParticleBinning diagnostics
Scanning for RadiationSpectrum diagnostics
Scanning for Performance diagnostics
Scanning for Screen diagnostics
Scanning for Tracked particle diagnostics
Scanning for new particle diagnostics
Fixed ROI mode: circle
  - left-click to place ROI center
  - arrow keys move ROI by 1 cell
  - Save stats -> case__roi_stats.txt in wkdir
  - Save figure -> case__Bx__t******fs__ROI__icx_icy.pdf in wkdir


Saved figure -> /Volumes/Irene/Sim_IFE_Rome/SG_l1LP_a0_500_ne_60_longVacuum_largeLy/SG_l1LP_a0_500_ne_60_longVacuum_largeLy__Bx__t000296fs__circle_r012__icxNA_icyNA.pdf


In [28]:
# do the temperal average for the spatial mean and std, and save the results in a txt file.

import numpy as np

def load_temporal_means(stats_file):
    """
    Reads roi_stats.txt and extracts:
        - spatial mean for each snapshot
        - spatial std for each snapshot
    Returns:
        means (array), stds (array)
    """
    means = []
    stds = []

    with open(stats_file, "r") as f:
        for line in f:
            if "mean=" in line:
                parts = line.strip().split("\t")
                for p in parts:
                    if p.startswith("mean="):
                        means.append(float(p.split("=")[1]))
                    if p.startswith("std="):
                        stds.append(float(p.split("=")[1]))

    means = np.array(means)
    stds = np.array(stds)

    return means, stds


def compute_temporal_average(means):
    """
    Returns:
        temporal mean
        temporal standard deviation
        standard error of the mean
    """
    N = len(means)
    mean_t = np.mean(means)
    std_t  = np.std(means, ddof=1)   # unbiased
    sem_t  = std_t / np.sqrt(N)

    return mean_t, std_t, sem_t


# # Example usage:
# stats_file = "/Users/yao/Documents/Data/IFE/ife_007_a0900_ne120_res80_ppc4/ife_007_a0900_ne120_res80_ppc4__roi_stats.txt"

# means, stds = load_temporal_means(stats_file)

# mean_case, std_temporal, sem_temporal = compute_temporal_average(means)

# print("Spatial means at each snapshot:", means)
# print("Temporal average (final data point):", mean_case)
# # print("Temporal std:", std_temporal)
# print("Standard error:", sem_temporal)

In [12]:
# Example usage:

stats_file = "/Users/yao/Documents/Data/IFE/LG_data1/LG_l1_Ltot2_CP_a600_avB/LG_l1_Ltot2_CP_a600_avB__roi_stats.txt"

means, stds = load_temporal_means(stats_file)

mean_case, std_temporal, sem_temporal = compute_temporal_average(means)

print("Spatial means at each snapshot:", means)
print("Temporal average (final data point):", mean_case)
# print("Temporal std:", std_temporal)
print("Standard error:", sem_temporal)

Spatial means at each snapshot: [-1.40495505 -1.37287112 -1.33604162 -1.28466645 -1.17140423]
Temporal average (final data point): -1.3139876932082775
Standard error: 0.04087693407335794


In [29]:
## do the analysis for all cases, and build a table of (a0, B_mean, error_bar_SEM) for plotting B vs a0 with error bars.


import os
import re
import csv
import numpy as np


# -----------------------------
# 1) Parsing helpers
# -----------------------------
def parse_a0_from_case_name(case_name: str) -> int:
    """
    Extract a0 from folder name patterns like:
      ife_007_a0900_ne120_res80_ppc4  -> a0 = 900
      something_a0300_...            -> a0 = 300
    """
    m = re.search(r"_a0?(\d+)", case_name)  # matches _a0900 or _a900
    if not m:
        raise ValueError(f"Could not parse a0 from case folder name: {case_name}")
    return int(m.group(1))


def find_stats_file(wkdir: str) -> str:
    """
    Prefer the case-named stats file: <case>__roi_stats.txt
    If not found, fall back to any *roi_stats*.txt in the folder.
    """
    case_name = os.path.basename(os.path.normpath(wkdir))
    preferred = os.path.join(wkdir, f"{case_name}__roi_stats.txt")
    if os.path.isfile(preferred):
        return preferred

    # fallback: search
    for fn in os.listdir(wkdir):
        if "roi_stats" in fn and fn.endswith(".txt"):
            return os.path.join(wkdir, fn)

    raise FileNotFoundError(f"No roi_stats txt file found in: {wkdir}")


def load_means_and_stds(stats_file: str):
    """
    Reads lines like:
      circle  t_fs=1234  icx=...  ... mean=... std=... ...
    Returns arrays of means, stds, and times (if present).
    """
    means, stds, times = [], [], []
    with open(stats_file, "r") as f:
        for line in f:
            line = line.strip()
            if not line or ("mean=" not in line):
                continue
            parts = line.split("\t")
            d = {}
            for p in parts:
                if "=" in p:
                    k, v = p.split("=", 1)
                    d[k.strip()] = v.strip()
            if "mean" in d:
                means.append(float(d["mean"]))
            if "std" in d:
                stds.append(float(d["std"]))
            if "t_fs" in d:
                # stored as integer fs in your writer
                try:
                    times.append(int(float(d["t_fs"])))
                except Exception:
                    pass

    means = np.asarray(means, dtype=float)
    stds = np.asarray(stds, dtype=float)
    times = np.asarray(times, dtype=int) if len(times) == len(means) else None
    return means, stds, times


# -----------------------------
# 2) Statistics per case
# -----------------------------
def summarize_case(means: np.ndarray):
    """
    Returns:
      B_mean  : temporal average of snapshot spatial means
      err_sem : SEM = std(means, ddof=1)/sqrt(N) (recommended error bar)
      std_t   : temporal std of the snapshot means (optional diagnostic)
      N       : number of snapshots used
    """
    means = np.asarray(means, dtype=float)
    N = means.size
    if N == 0:
        raise ValueError("No means found.")
    B_mean = float(np.mean(means))

    if N == 1:
        # With one snapshot you can't estimate variability; set err=0 or NaN
        std_t = float("nan")
        err_sem = float("nan")
    else:
        std_t = float(np.std(means, ddof=1))
        err_sem = std_t / np.sqrt(N)

    return B_mean, err_sem, std_t, int(N)


# -----------------------------
# 3) Main: build table for all cases
# -----------------------------
def build_summary(wkdirs, out_csv_path=None, print_ready_to_plot=True):
    rows = []

    for wkdir in wkdirs:
        case_name = os.path.basename(os.path.normpath(wkdir))
        a0 = parse_a0_from_case_name(case_name)

        stats_file = find_stats_file(wkdir)
        means, stds, times = load_means_and_stds(stats_file)

        B_mean, err_sem, std_t, N = summarize_case(means)

        row = {
            "case": case_name,
            "wkdir": wkdir,
            "a0": a0,
            "N_snapshots": N,
            "B_mean_1e5T": B_mean,
            "B_err_SEM_1e5T": err_sem,
            "B_std_temporal_1e5T": std_t,
            "stats_file": os.path.basename(stats_file),
        }
        rows.append(row)

    # sort by a0 (nice for plotting)
    rows.sort(key=lambda r: r["a0"])

    if print_ready_to_plot:
        print("\nReady-to-plot tuples: (a0, B_mean, error_bar_SEM)")
        for r in rows:
            print(f"({r['a0']}, {r['B_mean_1e5T']:.8g}, {r['B_err_SEM_1e5T']:.8g})")

    # write CSV
    if out_csv_path is not None:
        os.makedirs(os.path.dirname(out_csv_path), exist_ok=True)
        fieldnames = [
            "case", "a0", "N_snapshots",
            "B_mean_1e5T", "B_err_SEM_1e5T", "B_std_temporal_1e5T",
            "wkdir", "stats_file"
        ]
        with open(out_csv_path, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=fieldnames)
            w.writeheader()
            for r in rows:
                w.writerow(r)
        print(f"\nSaved summary CSV -> {out_csv_path}")

    return rows



In [30]:

# -----------------------------
# 4) Example usage
# -----------------------------
if __name__ == "__main__":
    wkdirs = [
           "/Users/yao/Documents/Data/IFE/LG_data1/LG_l1_Ltot2_CP_a600_avB/",
           "/Users/yao/Documents/Data/IFE/LG_data1/LG_l1_Ltot2_CP_a600_avB_NoRR/",
           "/Users/yao/Documents/Data/IFE/LG_data1/LG_l1_Ltot2_CP_ne50_a600_avB/",
           "/Users/yao/Documents/Data/IFE/LG_data1/LG_l1_Ltot2_CP_ne50_a600_avB_NoRR/",
         ]

    # Put CSV next to your cases (e.g. parent folder)
    parent = os.path.dirname(os.path.normpath(wkdirs[0]))
    out_csv = os.path.join(parent, "Bx_roi_summary_all_cases.csv")

    build_summary(wkdirs, out_csv_path=out_csv, print_ready_to_plot=True)


Ready-to-plot tuples: (a0, B_mean, error_bar_SEM)
(600, -1.3139877, 0.040876934)
(600, -1.2820727, 0.041984801)
(600, -1.3597163, 0.059455857)
(600, -1.3173842, 0.049442811)

Saved summary CSV -> /Users/yao/Documents/Data/IFE/LG_data1/Bx_roi_summary_all_cases.csv


In [31]:
# with the above prepared data file, we can now plot B_mean vs a0 with error bars representing the SEM over snapshots.

import os
import pandas as pd
import matplotlib.pyplot as plt

# ---- input CSV ----
csv_path = "/Users/yao/Documents/Data/IFE/LG_data1/Bx_roi_summary_all_cases.csv"
# If you want to run it from the attached file here, use:
# csv_path = "/mnt/data/Bx_roi_summary_all_cases.csv"

# ---- output figure ----
out_pdf = os.path.join(os.path.dirname(csv_path), "Bx_mean_vs_a0_errorbar.pdf")

# ---- load ----
df = pd.read_csv(csv_path)

# Ensure sorted by a0
df = df.sort_values("a0")

a0 = df["a0"].to_numpy()
B  = df["B_mean_1e5T"].to_numpy()
Be = df["B_err_SEM_1e5T"].to_numpy()

# ---- plot ----
fig, ax = plt.subplots(figsize=(6.5, 4.8))
ax.errorbar(a0, np.abs(B), yerr=Be, fmt="o-", capsize=4,
            markersize=8, color="blue", ecolor="blue", elinewidth=2)

ax.set_xlabel(r"$a_0$")
# ax.set_ylabel(r"$\langle B_x \rangle_{\mathrm{ROI}}$  $(10^5\,\mathrm{T})$")
ax.set_ylabel(r"$|\langle B_x \rangle_{\mathrm{ROI}}|$  $(10^5\,\mathrm{T})$")
ax.set_title(r"$\langle B_x \rangle$ vs $a_0$ (error bar = SEM over snapshots)")
ax.grid(True, which="both", alpha=0.3)

fig.tight_layout()
fig.savefig(out_pdf, dpi=600, bbox_inches="tight")
print(f"Saved: {out_pdf}")

plt.show()

Saved: /Users/yao/Documents/Data/IFE/LG_data1/Bx_mean_vs_a0_errorbar.pdf
